# Using MIMO Scaffold Functions in Jupyter Notebooks

This notebook demonstrates how to use the `mimo_pack.util.scaffold` functions directly from within Jupyter notebooks for creating session configuration files.

The scaffold utility provides programmatic access to session config generation, making it easy to integrate into analysis workflows and batch processing tasks.

## 1. Import Required Libraries

First, we need to import the scaffold functions and other necessary libraries.

In [1]:
import tempfile
import json
from pathlib import Path

# Import scaffold functions from the MIMO package
from mimo_pack.util.scaffold import create_session_config, init_multiple_sessions

# Import Session class for verification
from mimo_pack.core.session import Session

print("✓ Successfully imported scaffold functions!")
print("Available functions:")
print("  - create_session_config(): Create config for a single session")
print("  - init_multiple_sessions(): Batch create configs for multiple sessions")

✓ Successfully imported scaffold functions!
Available functions:
  - create_session_config(): Create config for a single session
  - init_multiple_sessions(): Batch create configs for multiple sessions


## 2. Create a Single Session Configuration

The `create_session_config()` function can be called directly from any notebook cell to create a configuration file for a single experimental session.

In [2]:
# Create a temporary directory to simulate a session directory
with tempfile.TemporaryDirectory() as temp_dir:
    session_path = Path(temp_dir) / "mouse01_session01"
    session_path.mkdir(exist_ok=True)
    
    print(f"Creating session config for: {session_path}")
    
    # Call the scaffold function directly
    config_path = create_session_config(
        session_dir=session_path,
        overwrite=False
    )
    
    print(f"\nConfig file created at: {config_path}")
    
    # Verify the config was created and is valid
    if config_path and config_path.exists():
        with open(config_path, 'r') as f:
            config_data = json.load(f)
            
        print(f"\n📋 Configuration preview:")
        print(f"   Keys: {list(config_data.keys())}")
        print(f"   File patterns: {list(config_data['paths'].keys())}")
        print(f"   Parameters: {list(config_data['parameters'].keys())}")
        print(f"   Metadata: {list(config_data['metadata'].keys())}")
    else:
        print("❌ Config creation failed")

Creating session config for: C:\Users\dbh60\AppData\Local\Temp\tmp1yke6eps\mouse01_session01
SUCCESS: Created session_config.json at: C:\Users\dbh60\AppData\Local\Temp\tmp1yke6eps\mouse01_session01\session_config.json

Generated configuration contains:
  - File patterns for SpikeGLX data (AP, LFP, NIDQ)
  - Default analysis parameters (LFP, CSD, evoked responses)
  - Metadata fields (subject, probe type, date, notes)

Next steps:
  1. Edit the config file to match your specific file naming
  2. Update analysis parameters as needed
  3. Fill in metadata fields
  4. Use Session class in your analysis scripts

Config file created at: C:\Users\dbh60\AppData\Local\Temp\tmp1yke6eps\mouse01_session01\session_config.json

📋 Configuration preview:
   Keys: ['paths', 'parameters', 'metadata']
   File patterns: ['ap_pattern', 'lfp_pattern', 'nidq_pattern']
   Parameters: ['lfp', 'csd', 'evoked_lfp']
   Metadata: ['subject', 'probe_type', 'date', 'notes']


## 3. Batch Create Multiple Session Configurations

In [ ]:
# Create multiple sessions at once
with tempfile.TemporaryDirectory() as temp_dir:
    experiment_dir = Path(temp_dir)
    
    # Create multiple session directories
    session_names = ['mouse01_session01', 'mouse01_session02', 'mouse02_session01']
    
    for session_name in session_names:
        (experiment_dir / session_name).mkdir(exist_ok=True)
    
    print(f"Batch creating configs for {len(session_names)} sessions...")
    
    # Use the batch initialization function
    created_configs = init_multiple_sessions(
        directory=experiment_dir,
        session_dirs=session_names,
        overwrite=False
    )
    
    print(f"\n✅ Successfully created {len(created_configs)} config files:")
    for i, config_path in enumerate(created_configs, 1):
        relative_path = config_path.relative_to(experiment_dir)
        print(f"   {i}. {relative_path}")
        
    # Verify they're all valid
    print(f"\n🔍 Verification:")
    for config_path in created_configs:
        if config_path.exists():
            print(f"   ✓ {config_path.parent.name}/session_config.json")
        else:
            print(f"   ❌ {config_path} not found")

## 4. Practical Example: Using Scaffold in Analysis Workflow

In [ ]:
# Example: Automating session setup in a research workflow
def setup_experimental_sessions(base_dir, animals, sessions_per_animal=3):
    """Set up session configs for a multi-animal experiment."""
    
    base_path = Path(base_dir)
    base_path.mkdir(exist_ok=True)
    
    all_configs = []
    
    for animal in animals:
        animal_sessions = []
        
        # Create session directories for this animal
        for session_num in range(1, sessions_per_animal + 1):
            session_name = f"{animal}_session{session_num:02d}"
            session_path = base_path / session_name
            session_path.mkdir(exist_ok=True)
            animal_sessions.append(session_name)
        
        # Create configs for this animal's sessions
        configs = init_multiple_sessions(
            directory=base_path,
            session_dirs=animal_sessions,
            overwrite=True  # Allow updates
        )
        
        all_configs.extend(configs)
        print(f"📁 Set up {len(configs)} sessions for {animal}")
    
    return all_configs

# Demo with synthetic experiment
animals = ['mouse01', 'mouse02', 'mouse03']

with tempfile.TemporaryDirectory() as temp_dir:
    print("🧪 Setting up experimental sessions...")
    
    config_files = setup_experimental_sessions(
        base_dir=temp_dir,
        animals=animals,
        sessions_per_animal=2
    )
    
    print(f"\n📋 Experiment setup complete!")
    print(f"   Total sessions: {len(config_files)}")
    print(f"   Animals: {len(animals)}")
    print(f"   Average sessions/animal: {len(config_files)/len(animals):.1f}")
    
    # Show the directory structure
    print(f"\n📂 Directory structure:")
    for path in sorted(Path(temp_dir).iterdir()):
        if path.is_dir():
            config_exists = (path / "session_config.json").exists()
            status = "✓" if config_exists else "❌"
            print(f"   {status} {path.name}/session_config.json")

## Summary

Yes, scaffold functions **absolutely can be called from Jupyter notebooks**! The examples above show:

### ✅ Key Benefits:
- **Programmatic Control**: Create configs with custom parameters and logic
- **Batch Processing**: Set up multiple sessions efficiently  
- **Integration**: Seamlessly fits into analysis workflows
- **Return Values**: Get file paths back for immediate use
- **Error Handling**: Catch and handle any issues gracefully

### 🔧 Common Use Cases:
1. **Experiment Setup**: Batch create configs for multi-animal studies
2. **Iterative Analysis**: Generate fresh configs during method development  
3. **Automated Workflows**: Part of larger data processing pipelines
4. **Custom Parameters**: Programmatically set specific analysis parameters

### 📝 Best Practices:
- Use `pathlib.Path` objects for cross-platform compatibility
- Handle return values (None indicates failure)
- Set `overwrite=True` when iterating during development
- Combine with other MIMO functions for complete workflows

The scaffold utility provides both **command-line convenience** and **programmatic flexibility**!